In [33]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
import re
import os
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage, SystemMessage, AIMessage


load_dotenv(override=True)



True

In [13]:
def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    pages = loader.load_and_split()
    content = ''
    for page in pages:
        content += page.page_content + "\n"
    return content
def clean_text(text):
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    return text
def split_chapters(text):
    pattern = r'(Chương\s+[IVXLCDM0-9]+.*?)((?=Chương\s+[IVXLCDM0-9]+)|$)'
    return re.findall(pattern, text, flags=re.DOTALL)


def split_articles(chapter_text):
    pattern = r'(Điều\s+\d+\.\s.*?)(?=(Điều\s+\d+\.|\Z))'
    return re.findall(pattern, chapter_text, flags=re.DOTALL)


def split_clauses(article_text):
    pattern = r'(\d+\.\s.*?)(?=(\d+\.|\Z))'
    return re.findall(pattern, article_text, flags=re.DOTALL)


def split_points(clause_text):
    pattern = r'([a-zđ]\)\s.*?)(?=([a-zđ]\)|\Z))'
    return re.findall(pattern, clause_text, flags=re.DOTALL)

In [14]:
structured_data = []
def parse_law(text,LAW_NAME):
    

    chapters = split_chapters(text)
    

    for chapter_full, _ in chapters:
        chapter_match = re.search(r'Chương\s+([IVXLCDM0-9]+)', chapter_full)
        chapter_number = chapter_match.group(1) if chapter_match else None

        articles = split_articles(chapter_full)

        for article_full, _ in articles:
            article_header_match = re.match(r'^Điều\s+(\d+)\.\s*(.+?)(?:\n|$)',article_full)
            if not article_header_match:
                continue

            article_number = int(article_header_match.group(1))
            article_title = article_header_match.group(2)

            clauses = split_clauses(article_full)

            if not clauses:
                structured_data.append({
                    "law_name": LAW_NAME,
                    "chapter": chapter_number,
                    "article": article_number,
                    "article_title": article_title,
                    "page_content": clean_text(article_full.strip())
                })
                continue

            for clause_full, _ in clauses:
                clause_match = re.match(r'(\d+)\.', clause_full)
                clause_number = int(clause_match.group(1)) if clause_match else None

                points = split_points(clause_full)

                if not points:
                    structured_data.append({
                        "law_name": LAW_NAME,
                        "chapter": chapter_number,
                        "article": article_number,
                        "article_title": article_title,
                        "clause": clause_number,
                        "page_content": clean_text(clause_full.strip())
                    })
                    continue

                for point_full, _ in points:
                    point_match = re.match(r'([a-zđ])\)', point_full)
                    point_letter = point_match.group(1) if point_match else None

                    structured_data.append({
                        "law_name": LAW_NAME,
                        "chapter": chapter_number,
                        "article": article_number,
                        "article_title": article_title,
                        "clause": clause_number,
                        "point": point_letter,
                        "page_content": clean_text(point_full.strip())
                    })

    return structured_data

In [15]:
for file_name in os.listdir('data/'):
        LAW_NAME = file_name.replace('.pdf', '')
        file_path = os.path.join('data/', file_name)
        pdf_content = load_pdf(file_path)
        # cleaned_content = clean_text(pdf_content)
        structured_law = parse_law(pdf_content, LAW_NAME)
        
        

In [16]:
len(structured_law)

3691

In [17]:
docs = []
for d in structured_law:

        doc = Document(
            page_content=d["page_content"],
            metadata={
                "law_name": d["law_name"],
                "chapter": d["chapter"],
                "article": d["article"],
                "article_title": d["article_title"],
                "clause": d.get("clause")
            }
        )

        docs.append(doc)

In [18]:
len(docs)

3691

In [3]:
# Pick an embedding model
db_name = "vectorstore"

embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding_v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


Chú ý :)))))

In [19]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 3691 documents


In [4]:
vectorstore = Chroma(
    persist_directory="./vectorstore",
    embedding_function=embeddings
)

In [5]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 3,691 vectors with 1,024 dimensions in the vector store


In [21]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['law_name'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange','yellow','black'][['luat_an_ninh_mang', 'luat_khcn', 'luat_ngan_sach_nha_nuoc', 'luat_quy_hoach','luat_thanh_tra','luat_xay_dung'].index(t)] for t in doc_types]

In [24]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [25]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [ ]:
retrieval = vectorstore.as_retriever(search_kwargs={"k": 5})
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0.3)

In [42]:
retrieval.invoke("AN ninh mạng là gì")

[Document(id='cd651236-0392-47cd-9a0d-4d971a453e67', metadata={'chapter': 'I', 'law_name': 'luat_an_ninh_mang', 'article': 2, 'clause': 1, 'article_title': 'Giải thích từ ngữ'}, page_content='1. An ninh mạng là sự ổn định, an ninh, an toàn của không gian mạng; bảo vệ hệ thống thông tin và bảo đảm thông tin, dữ liệu, hoạt động trên không gian mạng không gây phương hại đến an ninh quốc gia, trật tự, an toàn xã hội, quyền và lợi ích hợp pháp của cơ quan, tổ chức, cá nhân.'),
 Document(id='249a3ff3-cf5a-4e63-9905-148ac03ce26c', metadata={'article': 2, 'clause': 4, 'law_name': 'luat_an_ninh_mang', 'chapter': 'I', 'article_title': 'Giải thích từ ngữ'}, page_content='4. Bảo vệ an ninh mạng là phòng ngừa, phát hiện, ngăn chặn, xử lý hành vi xâm phạm an ninh mạng.'),
 Document(id='8b16e6fb-b877-443f-bf16-55d76b4665c5', metadata={'clause': 22, 'article': 2, 'article_title': 'Giải thích từ ngữ', 'law_name': 'luat_an_ninh_mang', 'chapter': 'I'}, page_content='22. Dịch vụ an ninh mạng là dịch vụ đư

In [35]:
SYSTEM_PROMPT_TEMPLATE = """Bạn là một trợ lý pháp lý chuyên nghiệp.

Nhiệm vụ của bạn:
- Trả lời câu hỏi dựa trên nội dung luật được cung cấp.
- Chỉ sử dụng thông tin có trong ngữ cảnh.
- Nếu không tìm thấy thông tin trong ngữ cảnh, hãy trả lời: "Không tìm thấy thông tin trong tài liệu được cung cấp."
- Trả lời rõ ràng, dễ hiểu, có trích dẫn Điều, Khoản nếu có.

Ngữ cảnh:
{context}

Trả lời:"""

In [ ]:
def answer_question(question: str, history):
    docs = retrieval.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content[0]["text"]

In [52]:
print(answer_question("An ninh mạng là gì?", []))


KeyboardInterrupt

